# Comparing OpenTofu and Terraform state management

A focused comparison of how each tool handles local and remote state, state locking, migration paths, and the operational differences that matter when moving between them. This notebook uses scratch directories for file-level experiments; no cloud credentials are required.

## Purpose

Both OpenTofu and Terraform store state as JSON in `terraform.tfstate` (local) or via backend blocks (remote) with the same schema. The migration path — copy the working directory, re-init with the other binary, verify an empty plan — works because the state format is identical. This notebook demonstrates the shared state structure, shows how backend configuration translates, and outlines the practical steps for migrating state between tools.

The comparison is written from a reference perspective; the file-level experiments run in throwaway directories and can be executed without real cloud resources.

## Setup

Both CLIs (`terraform` and `tofu`) should be on PATH for optional binary checks. All file operations use temporary directories under `/tmp` so nothing touches production state.

In [ ]:
# last_verified: 2026-10-06 · OpenTofu/Terraform state management comparison
import hashlib
import json
import shutil
import subprocess
from pathlib import Path

tf_dir = Path("/tmp/ot-state-compare-terraform")
ot_dir = Path("/tmp/ot-state-compare-opentofu")
for d in (tf_dir, ot_dir):
    d.mkdir(parents=True, exist_ok=True)
print(f"terraform dir: {tf_dir}")
print(f"opentofu dir:  {ot_dir}")

In [ ]:
# Confirm both binaries respond (informational — file cells below run either way)
for binary in ("terraform", "tofu"):
    try:
        result = subprocess.run([binary, "version"], capture_output=True, text=True, timeout=30)
        first = result.stdout.strip().splitlines()[0] if result.stdout.strip() else "no output"
    except FileNotFoundError:
        first = "not on PATH — skipping binary checks"
    print(f"{binary}: {first}")

## Step 1 — local state structure

Both tools write `terraform.tfstate` as a JSON document with the same top-level keys: `version`, `terraform_version`, `serial`, `lineage`, `outputs`, `resources`, and `check_results`. The schema is what makes cross-tool state portability possible.

In [ ]:
# Create a minimal valid state file in each directory to inspect the schema
minimal_state = {
    "version": 4,
    "terraform_version": "1.0.0",
    "serial": 1,
    "lineage": "test-lineage",
    "outputs": {},
    "resources": [],
    "check_results": null
}

for d in (tf_dir, ot_dir):
    with open(d / "terraform.tfstate", "w") as f:
        json.dump(minimal_state, f, indent=2)
print("Created identical minimal terraform.tfstate in both directories")
print(json.dumps(minimal_state, indent=2))

In [ ]:
# Verify the two state files are byte-identical
def sha(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:12]

tf_sum = sha(tf_dir / "terraform.tfstate")
ot_sum = sha(ot_dir / "terraform.tfstate")
print(f"terraform state: {tf_sum}")
print(f"opentofu state:  {ot_sum}")
print("identical:", tf_sum == ot_sum)

## Step 2 — remote backend configuration

Remote state is configured with a `backend` block inside the `terraform` block. The block shape and argument names are the same for both tools. The example below uses the S3 backend; other backends (azurerm, gcs, kubernetes, http, pg, etc.) follow the same pattern.

In [ ]:
# Backend configuration shape is identical for both tools (shown, not applied).
# Replace placeholder values before using with a real backend.
backend_s3 = """
terraform {
  backend "s3" {
    bucket         = "my-terraform-state-bucket"
    key            = "env/prod/terraform.tfstate"
    region         = "us-east-1"
    encrypt        = true
    dynamodb_table = "terraform-lock-table"
  }
}
"""

for d in (tf_dir, ot_dir):
    with open(d / "backend.tf", "w") as f:
        f.write(backend_s3)
print(backend_s3)
print("backend block written to both directories")

### Backend arguments comparison

| Argument | Purpose | Notes |
|---|---|---|
| `bucket` | S3 bucket name | Required |
| `key` | Object key (state path) | Required |
| `region` | AWS region | Required |
| `encrypt` | Server-side encryption | Optional, recommended |
| `dynamodb_table` | DynamoDB table for state locking | Optional, required for locking |
| `role_arn` | IAM role for cross-account access | Optional |
| `profile` | AWS CLI profile name | Optional |

The same argument set applies under both `terraform` and `tofu`. No translation is needed when switching tools.

## Step 3 — state locking behavior

Both tools implement state locking through the backend. For the S3 backend, locking uses a DynamoDB table specified by `dynamodb_table`. The lock mechanism is backend-native, not tool-native — the same DynamoDB table works regardless of whether `terraform` or `tofu` acquires the lock.

In [ ]:
# Demonstrate that lock table configuration is identical
lock_config = """
# DynamoDB table for state locking (Terraform/OpenTofu agnostic)
# Table name: terraform-lock-table
# Partition key: LockID (string)
# TTL attribute: TTL (number, optional but recommended)
"""
print(lock_config)
print("Lock table schema is independent of the IaC tool.")

## Step 4 — migration path: copy-and-replan

The standard migration procedure:

1. **Copy** the working directory (configuration + state) to a new location.
2. **Re-init** with the target binary (`tofu init` or `terraform init`). This re-resolves providers and refreshes the lock file.
3. **Plan** with the target binary and verify the plan is empty (no changes).
4. **Cut over** by pointing CI/CD and team workflows to the new directory/binary.
5. **Retire** the original directory after a validation window.

In [ ]:
# Rehearse the copy half of copy-and-replan with plain file operations
migrated = Path("/tmp/ot-state-compare-migrated")
if migrated.exists():
    shutil.rmtree(migrated)
shutil.copytree(tf_dir, migrated)

orig = sorted(p.name for p in tf_dir.iterdir())
copy = sorted(p.name for p in migrated.iterdir())
print(f"original files: {orig}")
print(f"migrated files: {copy}")
print("copy complete:", orig == copy)
print()
print("Next in real use: run 'tofu init' in the copy so providers resolve,")
print("then 'tofu plan' and expect no changes before cutting over.")

## Step 5 — state operations comparison

Common state operations work identically under both tools because the CLI verbs and flags are the same.

In [ ]:
# State command parity table
state_commands = [
    ("terraform state list", "tofu state list"),
    ("terraform state show <resource>", "tofu state show <resource>"),
    ("terraform state mv <src> <dst>", "tofu state mv <src> <dst>"),
    ("terraform state rm <resource>", "tofu state rm <resource>"),
    ("terraform state pull", "tofu state pull"),
    ("terraform state push <file>", "tofu state push <file>"),
    ("terraform state replace-provider <old> <new>", "tofu state replace-provider <old> <new>"),
]

print("State command parity:")
for tf_cmd, ot_cmd in state_commands:
    print(f"  terraform: {tf_cmd}")
    print(f"  opentofu:  {ot_cmd}")
    print()

## Step 6 — CI/CD integration for state

Pipeline stages that interact with state (plan, apply, state operations) use the same command structure. Only the binary name differs.

In [ ]:
# CI stage sequence for state operations differs only in binary name
def pipeline(binary):
    return [
        [binary, "init"],
        [binary, "plan", "-out=tfplan"],
        # Review gate here
        [binary, "apply", "tfplan"],
    ]

tf_pipe = pipeline("terraform")
ot_pipe = pipeline("tofu")
for t, o in zip(tf_pipe, ot_pipe):
    print("terraform: " + " ".join(t))
    print("opentofu:  " + " ".join(o))
    print()

tf_shape = [[c for c in cmd[1:]] for cmd in tf_pipe]
ot_shape = [[c for c in cmd[1:]] for cmd in ot_pipe]
print("Same stage shape (binary name aside):", tf_shape == ot_shape)

## Head-to-head: state management

| Aspect | Terraform | OpenTofu |
|---|---|---|
| Local state file | `terraform.tfstate` (JSON) | Same path, same JSON schema |
| State version | Version 4 (current) | Same versioning scheme |
| Remote backend block | Inside `terraform { backend "type" { ... } }` | Identical block shape |
| S3 backend arguments | bucket, key, region, encrypt, dynamodb_table, role_arn, profile, ... | Same argument set |
| State locking | Backend-native (DynamoDB for S3) | Same backend, same lock table |
| Lock ID format | `bucket/key` | Same format |
| `terraform state` commands | list, show, mv, rm, pull, push, replace-provider | Same verbs, same flags |
| `terraform import` | `terraform import <addr> <id>` | `tofu import <addr> <id>` |
| State migration | Copy dir, re-init, empty-plan check | Same procedure |
| Workspace support | `terraform workspace` | `tofu workspace` (same subcommands) |
| Partial state | `-target` applies to state operations | Same flag behavior |

## Verify

To confirm the state management comparison on your own machine:

1. Diff the two `backend.tf` files — expect no differences.
2. With a real S3 bucket and DynamoDB table configured, run `terraform init` in one directory and `tofu init` in the other — both should configure the same backend and acquire locks from the same DynamoDB table.
3. For the migration path, copy a directory with real state, run `tofu init` then `tofu plan` in the copy — expect an empty plan before cutting over.
4. Run `terraform state list` and `tofu state list` against the same state file — output should be identical.

## What to explore next

Test the migration path with a real remote backend that has active state locking, and with workspaces in use, to verify the copy-and-replan procedure holds under concurrent access scenarios. The local file-level experiment above covers the schema and configuration parity; the operational differences emerge under lock contention and multi-environment workflows.

In [ ]:
# Clean up the scratch runs (local-only dirs created by this notebook)
import shutil as _shutil

for _d in (tf_dir, ot_dir, Path("/tmp/ot-state-compare-migrated")):
    _shutil.rmtree(_d, ignore_errors=True)
print("scratch dirs removed")